# Benchmark dei 15 modelli su telemetria SolarTech LabEstensione di `solar_telemetry.ipynb`. Prende le **stesse finestre** di telemetria misurata e lemanda a tutti i checkpoint del sweep piu' al Chronos-Bolt pubblicato, poi riporta MAE, bias,varianza dell'errore e weighted quantile loss in una tabella sola.### Cosa misura, e cosa non misuraIl baseline della tabella e' `amazon/chronos-bolt-tiny` **come rilasciato**. Va letto sapendo cosasepara quel checkpoint dagli altri quindici: e' preaddestrato su un corpus osservativo di ordine$10^7$ serie *che comprende energia e meteo*, mentre i quindici hanno visto solo la mixturesintetica del sweep. Il divario verso il baseline e' quindi **differenza di corpus diaddestramento**, non differenza di geometria, e non e' un risultato sull'aliasing.Per questo ogni riga porta **due** colonne di scarto:| colonna | confronto | cosa isola ||---|---|---|| `d_MAE_vs_base` | verso `chronos-bolt-tiny` pubblicato | corpus **e** geometria insieme || `d_MAE_vs_1616` | verso `p16-s16-seed42` retrainato | **solo** la geometria, a parita' di corpus, step e seed |La seconda e' l'unica delle due che isola la variabile del progetto: `p16-s16` e' la geometriastock di Bolt, addestrata esattamente come le altre quattordici. Se in discussione qualcuno chiede"e a parita' di dati?", la risposta e' quella colonna.### Il canaleSi usa **`G_h`**, l'irradianza orizzontale, non `PV_Power`. La ragione e' aritmetica: `PV_Power` e'`NaN` fuori dalle ore di luce e il suo blocco contiguo piu' lungo in tutto il file e' di **880campioni**, contro i $2048 + 64 = 2112$ che servono per una finestra a contesto pieno. Su`PV_Power` il numero di finestre utilizzabili a questo contesto e' **zero**. `G_h` e' valido al99,1%, e' il driver fisico della potenza, e lascia il contesto alla lunghezza su cui i modelli sonostati addestrati.

## 0, Configurazione

In [ ]:
# --------------------------------------------------------------------------------------- #
#  CONFIGURAZIONE
# --------------------------------------------------------------------------------------- #
CSV_PATH  = "../data/dataset/Dataset-SolarTechLab.csv"
CANALE    = "G_h"            # G_h | G_tilt | T_air | W_s   (PV_Power non regge il contesto 2048)

CTX, PRED = 2048, 64         # le lunghezze del training del sweep (train_sweep.py)
L_FIN     = CTX + PRED       # 2112 campioni contigui per finestra

N_FINESTRE = 150             # finestre DISGIUNTE. Piu' basso e' il numero, piu' margine ha
                             # ciascun blocco per centrare l'ora del taglio: a 150 il margine
                             # e' di circa 23 ore e la copertura oraria e' quasi esatta.
                             # Il massimo che il file concede a questo contesto e' ~207.
SEED       = 42
N_BOOT     = 2000            # ricampionamenti per gli intervalli sugli scarti appaiati

BASE_ID     = "amazon/chronos-bolt-tiny"   # il baseline della tabella, come rilasciato
RIF_INTERNO = (16, 16)                     # il riferimento a parita' di corpus, retrainato

BATCH   = 64
DEVICE  = None               # None = cuda se c'e'
OUT_DIR = "_run/solar_bench"

# Predittore finto al posto dei modelli: serve a provare campionamento, metriche e bootstrap
# senza GPU e senza checkpoint. NON produce risultati, e la tabella lo dichiara.
MODO_PROVA = False           #@param {type:"boolean"}

# Soglie di regime, in gradi di elevazione solare, per la ripartizione in coda al notebook.
ELEV_GIORNO = 10.0
SITE_LAT, SITE_LON, SITE_TZ = 45.502, 9.156, 1.0     # SolarTech Lab, Politecnico di Milano

## 1, Caricamento e pulizia

In [ ]:
import json, time, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

RAW = pd.read_csv(CSV_PATH, sep=";", parse_dates=["Time"], dayfirst=True)
RAW = RAW.replace(-999999.0, np.nan)
assert CANALE in RAW.columns, f"{CANALE} non fra {list(RAW.columns)}"

Y_ALL = RAW[CANALE].to_numpy(float)
OK    = np.isfinite(Y_ALL)
T_TOT = len(Y_ALL)
print(f"{T_TOT:,} righe   {RAW.Time.min().date()} -> {RAW.Time.max().date()}")
print(f"{CANALE}: {OK.sum():,} validi ({100*OK.mean():.1f}%)   "
      f"intervallo {np.nanmin(Y_ALL):.4g} .. {np.nanmax(Y_ALL):.4g}")

def _elevazione(ts, lat=SITE_LAT, lon=SITE_LON, tz=SITE_TZ):
    """Elevazione solare in gradi (equazioni NOAA a bassa precisione), come in solar_telemetry."""
    t = pd.DatetimeIndex(ts); utc = t - pd.Timedelta(hours=tz)
    n  = utc.dayofyear.to_numpy(float)
    hr = utc.hour.to_numpy(float) + utc.minute.to_numpy(float)/60 + utc.second.to_numpy(float)/3600
    g  = 2*np.pi/365.0*(n - 1 + (hr - 12)/24.0)
    eq = 229.18*(0.000075 + 0.001868*np.cos(g) - 0.032077*np.sin(g)
                 - 0.014615*np.cos(2*g) - 0.040849*np.sin(2*g))
    dec = (0.006918 - 0.399912*np.cos(g) + 0.070257*np.sin(g) - 0.006758*np.cos(2*g)
           + 0.000907*np.sin(2*g) - 0.002697*np.cos(3*g) + 0.00148*np.sin(3*g))
    tst = (hr*60 + eq + 4*lon) % 1440
    ha  = np.deg2rad(tst/4.0 - 180.0); la = np.deg2rad(lat)
    return np.rad2deg(np.arcsin(np.clip(np.sin(la)*np.sin(dec)
                                        + np.cos(la)*np.cos(dec)*np.cos(ha), -1, 1)))

ELEV = _elevazione(RAW["Time"])

## 2, Le finestreDue requisiti, e sono in tensione fra loro.**Disgiunte.** Due finestre distanti un minuto condividono $2047$ campioni di contesto su $2048$:campionare minuti a caso produce molte osservazioni e pochissima informazione indipendente. Qui lefinestre non si toccano, quindi la finestra **e'** l'unita' indipendente e il bootstrap piu' avantinon ha bisogno di blocchi.**A ore diverse.** Ogni finestra deve prevedere da un momento del giorno diverso, altrimenti ilconfronto fra modelli si riduce a un solo tipo di transizione. L'ora che conta e' quella del**taglio** — il primo punto previsto — non quella in cui comincia il contesto.**La tensione.** Il limite superiore di finestre disgiunte e' $\lfloor 525601/2112 \rfloor = 248$, etenendo conto dei buchi del logger circa $207$. Ma piu' finestre si chiedono, meno spazio resta perscegliere *dove* metterle: a $230$ il margine dentro un blocco e' di $165$ minuti, cioe' due o treore di taglio possibili, e l'istogramma orario esce sbilanciato. A $N = 150$ il margine e' di circa$23$ ore e l'ora del taglio si puo' scegliere quasi liberamente. Il campionamento parte da $150$ perquesto; il parametro resta libero e la cella stampa l'istogramma che ne esce.**Come si campiona.** Non a posizioni casuali: piazzare intervalli a caso su una retta e scartarele sovrapposizioni si ferma alla costante di Rényi, circa il $74{,}8\%$ di riempimento. Si usa un**campionamento sistematico con bilanciamento orario**: il file e' diviso in $N$ blocchi uguali,i blocchi si percorrono in ordine casuale, e dentro ciascuno si sceglie fra le posizioni validequella la cui ora di taglio e' al momento **la meno rappresentata**. Il blocco garantisce lacopertura dell'anno, la scelta dentro il blocco appiana le ventiquattro ore, e la disgiunzione e'per costruzione. Una seconda passata riempie gli spazi lasciati dai blocchi caduti su un buco.**Quello che non si fa, e perche'.** Le ultime finestre fino a $207$ si otterrebbero piastrellandoi blocchi validi con finestre adiacenti, a passo esattamente $L = 2112$ minuti. Ma$2112/1440 = 22/15$, quindi un campionatore cosi' ripete **lo stesso orario ogni quindicifinestre**: un pettine a passo fisso contro il ciclo diurno, cioe' esattamente l'errore che questoprogetto studia, commesso sul disegno sperimentale invece che nel modello. La cella stampa ilconfronto fra i due passi, in orari distinti visitati prima di ripetersi, cosi' la scelta si vedeinvece di doverla credere.

In [ ]:
# finestra valida = tutti i suoi L_FIN campioni sono presenti. Test O(1) via somma cumulata.
_cs = np.concatenate([[0], np.cumsum(OK)])
VALIDA = (_cs[L_FIN:] - _cs[:-L_FIN]) == L_FIN        # VALIDA[s] per lo start s
print(f"posizioni di partenza valide: {VALIDA.sum():,} su {len(VALIDA):,}")
print(f"massimo teorico di finestre disgiunte: {T_TOT // L_FIN}")

def campiona_disgiunte(valida, n, L, seed, ctx=CTX):
    """N finestre disgiunte, distribuite sulle ventiquattro ore del giorno.

    Passata 1, sistematica e bilanciata. Il file e' diviso in n blocchi uguali e ogni blocco
    ospita una finestra. Il margine di manovra dentro un blocco e' (blocco - L) minuti, che
    copre due o tre ore di taglio diverse: fra le posizioni valide del blocco si sceglie quella
    la cui ORA DI TAGLIO e' al momento la meno rappresentata. Il blocco garantisce la copertura
    dell'anno, la scelta dentro il blocco appiana l'istogramma orario, e le finestre restano
    disgiunte perche' i blocchi lo sono.

    Passata 2, riempimento. Un blocco il cui margine cade dentro un buco del logger non produce
    niente. Si percorrono allora gli spazi fra finestre accettate e, dove ci sta una finestra
    intera, se ne piazza una con lo stesso criterio orario.

    L'ora e' quella del TAGLIO - il primo punto previsto - non quella dell'inizio del contesto:
    e' l'istante da cui ogni modello sta effettivamente prevedendo.
    """
    import bisect
    rng = np.random.default_rng(seed)
    T = len(valida)
    blocco = T // n
    if blocco < L:
        raise ValueError(f"blocco {blocco} < finestra {L}: N_FINESTRE troppo alto per questo file")

    conta = np.zeros(24, dtype=int)
    acc = []

    def _ora(s): return ((s + ctx) % 1440) // 60

    def _libera(s):
        i = bisect.bisect_left(acc, s)
        if i < len(acc) and acc[i] - s < L: return False
        if i > 0 and s - acc[i-1] < L: return False
        return True

    def _prendi(lo, hi, controlla_libera):
        """Fra le posizioni valide in [lo, hi], una dell'ora meno rappresentata. None se nessuna."""
        if hi < lo: return None
        amm = lo + np.flatnonzero(valida[lo:hi + 1])
        if controlla_libera and amm.size:
            amm = np.array([s for s in amm if _libera(int(s))], dtype=int)
        if amm.size == 0: return None
        ore = ((amm + ctx) % 1440) // 60
        c = conta[ore]
        migliori = amm[c == c.min()]
        return int(rng.choice(migliori))

    ordine = rng.permutation(n)              # i blocchi in ordine casuale, cosi' nessuna ora
    for k in ordine:                         # viene privilegiata dal solo scorrere del tempo
        s = _prendi(int(k)*blocco, min(int(k)*blocco + blocco - L, T - 1), False)
        if s is not None:
            bisect.insort(acc, s); conta[_ora(s)] += 1
    n1 = len(acc)

    while len(acc) < n:
        bordi = [-L] + acc + [T + L]
        spazi = [(bordi[i] + L, min(bordi[i+1] - L, T - 1)) for i in range(len(bordi) - 1)]
        spazi = [(a, b) for a, b in spazi if b >= a]
        aggiunte = 0
        for a, b in sorted(spazi, key=lambda ab: ab[1] - ab[0], reverse=True):
            if len(acc) >= n: break
            s = _prendi(a, b, True)
            if s is not None:
                bisect.insort(acc, s); conta[_ora(s)] += 1; aggiunte += 1
        if aggiunte == 0: break

    print(f"  passata sistematica: {n1}/{n}   riempimento degli spazi: +{len(acc)-n1}")
    if len(acc) < n:
        print(f"  il file non concede {n} finestre disgiunte valide: ottenute {len(acc)}")
    print(f"  finestre per ora del taglio: min {conta.min()}, max {conta.max()}, "
          f"ore mai visitate {(conta == 0).sum()}")
    return np.array(acc, dtype=int)


def _diagnosi_risonanza(passo, etichetta):
    """Quanti orari distinti visita un campionatore a passo fisso prima di ripetersi."""
    from math import gcd
    avanzo = passo % 1440
    distinti = 1440 // gcd(avanzo, 1440) if avanzo else 1
    print(f"  {etichetta:36s} passo {passo:6d} min -> avanzo {avanzo:4d} min/finestra, "
          f"{distinti:4d} orari distinti")
    return distinti

print("\nrisonanza col ciclo diurno (1440 min):")
_d_sis = _diagnosi_risonanza(len(VALIDA)//N_FINESTRE, "sistematico con jitter (usato)")
_d_pia = _diagnosi_risonanza(L_FIN, "piastrellatura adiacente (scartata)")
if _d_pia >= _d_sis:
    print("  ATTENZIONE: la piastrellatura non sarebbe peggiore, rivedere la scelta")

START = campiona_disgiunte(VALIDA, N_FINESTRE, L_FIN, SEED)
N = len(START)
assert np.all(np.diff(START) >= L_FIN), "le finestre si sovrappongono"
print(f"\nfinestre ottenute: {N}   disgiunte: si   passo minimo {np.diff(START).min()} >= {L_FIN}")

# contesti e veri. Con N di quest'ordine sta tutto in memoria: N x 2112 x 8 byte ~ 4 MB.
IDX  = START[:, None] + np.arange(L_FIN)[None, :]
CTXS = Y_ALL[IDX[:, :CTX]].astype(np.float32)          # [N, CTX]
YTRUE = Y_ALL[IDX[:, CTX:]].astype(np.float64)         # [N, PRED]
TCUT = RAW["Time"].to_numpy()[START + CTX]             # istante del primo punto previsto
assert np.isfinite(CTXS).all() and np.isfinite(YTRUE).all()

# regime, dall'elevazione solare sul solo TARGET (nessuna informazione dal futuro del canale)
_el = ELEV[IDX[:, CTX:]]
REGIME = np.where(_el.max(1) <= 0, "notte",
          np.where(_el.min(1) >= ELEV_GIORNO, "giorno", "transizione"))

FIN = pd.DataFrame({
    "finestra": np.arange(N), "start": START, "t_taglio": TCUT,
    "ora": pd.DatetimeIndex(TCUT).hour + pd.DatetimeIndex(TCUT).minute/60,
    "doy": pd.DatetimeIndex(TCUT).dayofyear, "regime": REGIME,
    "elev_max": _el.max(1).round(2), "y_media": YTRUE.mean(1).round(2),
    "y_max": YTRUE.max(1).round(2), "ctx_media": CTXS.mean(1).round(2)})
print("\nregimi:"); print(FIN.regime.value_counts().to_string())
print(f"copertura annuale: giorni {FIN.doy.min()} .. {FIN.doy.max()}, {FIN.doy.nunique()} distinti")
_orari = FIN.ora.astype(int).value_counts().reindex(range(24), fill_value=0)
print("finestre per ora del taglio, dalle 00 alle 23:")
print("  " + " ".join(f"{int(v):2d}" for v in _orari.to_numpy()))
print(f"  min {_orari.min()}, max {_orari.max()}, ore mai visitate {int((_orari == 0).sum())}")
assert _orari.min() >= 1, "c'e' un'ora del giorno da cui nessuna finestra prevede"
FIN.to_csv(f"{OUT_DIR}/finestre.csv", index=False)
display(FIN.head(8))

### Il manifestoOgni risultato salvato porta con se' le condizioni in cui e' stato prodotto. Se una ripresasuccessiva cambia canale, contesto, numero di finestre o seme, le previsioni in cache non sono piu'confrontabili e vanno rifatte: il controllo qui sotto se ne accorge invece di mescolare due run.

In [ ]:
MANIFESTO = dict(canale=CANALE, ctx=CTX, pred=PRED, n_finestre=int(N), seed=SEED,
                 csv_sha=hashlib.sha256(Path(CSV_PATH).read_bytes()).hexdigest()[:16],
                 start_sha=hashlib.sha256(START.tobytes()).hexdigest()[:16])
_mp = Path(f"{OUT_DIR}/manifesto.json")
if _mp.exists():
    _vecchio = json.loads(_mp.read_text())
    _diff = {k: (_vecchio.get(k), v) for k, v in MANIFESTO.items() if _vecchio.get(k) != v}
    if _diff:
        print("MANIFESTO DIVERSO dalla cache presente. Le previsioni salvate NON sono riusabili:")
        for k, (a, b) in _diff.items(): print(f"   {k}: in cache {a!r}, ora {b!r}")
        print("   -> tutti i modelli verranno rifatti e i pred_*.npy sovrascritti")
        RIUSA_CACHE = False
        # Il manifesto va riscritto ORA, non a fine notebook: se la sessione cade a meta' loop,
        # su disco ci sarebbero previsioni nuove sotto un manifesto vecchio, e la ripresa
        # successiva le riuserebbe credendole valide. Con RIUSA_CACHE False tutti e sedici i
        # file vengono comunque riscritti, quindi allineare subito il manifesto e' corretto.
        _mp.write_text(json.dumps(MANIFESTO, indent=2))
    else:
        print("manifesto identico alla cache: le previsioni gia' calcolate verranno riusate")
        RIUSA_CACHE = True
else:
    _mp.write_text(json.dumps(MANIFESTO, indent=2)); RIUSA_CACHE = True
print(json.dumps(MANIFESTO, indent=2))

## 3, I modelliI quindici sono `probe_lib.DELIVERABLE3_MODELS`, cioe' le geometrie del sweep che chiudono ilcontesto ($480 \bmod S = 0$) al netto delle esclusioni dichiarate. E' la stessa popolazione su cuigira l'analisi bayesiana, quindi la tabella e i modelli parlano delle stesse quindici cose.Il sedicesimo e' `amazon/chronos-bolt-tiny` come rilasciato. Compare con l'etichetta`base (pubblicato)` e non e' una geometria del sweep: e' il riferimento esterno.

In [ ]:
if MODO_PROVA:
    GEOMS = [(8,8),(16,8),(16,12),(16,16),(24,8),(24,12),(24,16),(24,20),(24,24),
             (32,8),(32,12),(32,16),(32,20),(32,24),(32,32)]
    print("MODO PROVA: geometrie hardcoded, nessun checkpoint caricato")
else:
    import probe_lib as pl
    GEOMS = list(pl.DELIVERABLE3_MODELS)

assert RIF_INTERNO in GEOMS, f"{RIF_INTERNO} non e' fra le geometrie: manca il riferimento interno"
print(f"{len(GEOMS)} geometrie del sweep + 1 baseline pubblicato = {len(GEOMS)+1} righe")
print("  ", ", ".join(f"p{P}-s{S}" for P, S in GEOMS))

def tag(P, S): return f"p{P}-s{S}"
TAG_BASE = "base (pubblicato)"
TAG_RIF  = tag(*RIF_INTERNO)
RIGHE = [(TAG_BASE, None)] + [(tag(P, S), (P, S)) for P, S in GEOMS]

## 4, Le previsioniOgni modello riceve **le stesse** `N` finestre, nello stesso ordine, e restituisce tutti e nove iquantili. Il file `pred_<tag>.npy` che ne esce ha forma `[N, 9, PRED]` e pesa meno di un megabyte,quindi le previsioni si conservano tutte e le metriche si possono ricalcolare senza rifare i fit.Un checkpoint per volta: si carica, si prevede, si libera la memoria della GPU. Le previsioni gia'in cache si saltano.

In [ ]:
QUANTILI = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
QI_MED   = QUANTILI.index(0.5)

def _predittore_finto(nome, seed):
    """Persistenza + deriva, con un errore che dipende dal tag. Solo per provare la pipeline."""
    rng = np.random.default_rng(abs(hash(nome)) % (2**31) + seed)
    scala = 1.0 + 0.10*rng.standard_normal()
    def f(X):
        ultimo = X[:, -1:].astype(np.float64)
        media  = X[:, -240:].mean(1, keepdims=True).astype(np.float64)
        base   = ultimo + (media - ultimo)*np.linspace(0, 1, PRED)[None, :]
        base   = base*scala + rng.normal(0, 8.0, size=(len(X), PRED))
        larg   = np.abs(base).mean()*0.25 + 1.0
        off    = np.array([-1.28,-0.84,-0.52,-0.25,0,0.25,0.52,0.84,1.28])[None,:,None]*larg
        return np.clip(base[:, None, :] + off, 0, None)
    return f

def _carica(geom, dev):
    """Pipeline e etichetta. geom None = il pubblicato, altrimenti il retrainato di quel (P,S)."""
    from chronos import BaseChronosPipeline
    if geom is None:
        return BaseChronosPipeline.from_pretrained(BASE_ID, device_map=dev), BASE_ID
    import model_loader as ml
    P, S = geom
    ck = ml.resolve_local_checkpoint(P, S)
    if ck is not None:
        return BaseChronosPipeline.from_pretrained(str(ck), device_map=dev), f"local {ck.name}"
    return (BaseChronosPipeline.from_pretrained(ml.SWEEP_REPO, subfolder=f"p{P}-s{S}-seed42",
                                                revision=ml.SWEEP_REVISION, device_map=dev),
            f"hub {ml.SWEEP_REPO[:28]}@{ml.SWEEP_REVISION[:8]}")

def prevedi_tutto(nome, geom):
    f = Path(f"{OUT_DIR}/pred_{nome.replace(' ','_').replace('(','').replace(')','')}.npy")
    if RIUSA_CACHE and f.exists():
        P_ = np.load(f)
        if P_.shape == (N, len(QUANTILI), PRED):
            print(f"  {nome:20s} da cache"); return P_
    t0 = time.time()
    if MODO_PROVA:
        out = _predittore_finto(nome, SEED)(CTXS); etichetta = "FINTO"
    else:
        import torch
        dev = DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
        pipe, etichetta = _carica(geom, dev)
        cfg = pipe.model.config.chronos_config
        if geom is not None and (int(cfg["input_patch_size"]), int(cfg["input_patch_stride"])) != geom:
            raise ValueError(f"{nome}: il checkpoint dichiara una geometria diversa")
        qs = list(cfg["quantiles"])
        sel = [qs.index(q) for q in QUANTILI] if all(q in qs for q in QUANTILI) else list(range(len(qs)))
        blocchi = []
        with torch.no_grad():
            for i in range(0, N, BATCH):
                xb = torch.tensor(CTXS[i:i+BATCH], device=dev)
                yb = pipe.predict(xb, prediction_length=PRED)      # [b, Q, PRED]
                blocchi.append(yb[:, sel, :].float().cpu().numpy())
        out = np.concatenate(blocchi, 0).astype(np.float64)
        del pipe
        if dev == "cuda": torch.cuda.empty_cache()
    assert out.shape == (N, len(QUANTILI), PRED), out.shape
    np.save(f, out)
    print(f"  {nome:20s} {etichetta:44s} {time.time()-t0:6.1f}s")
    return out

print(f"previsioni su {N} finestre, {len(RIGHE)} modelli"
      + ("   [MODO PROVA]" if MODO_PROVA else ""))
PRED_Q = {}
for nome, geom in RIGHE:
    PRED_Q[nome] = prevedi_tutto(nome, geom)
print("fatto.")

## 5, Le metriche**MAE, bias, varianza** sono calcolate sull'errore puntuale $e = \hat y_{0.5} - y$, con $\hat y_{0.5}$la mediana dei nove quantili — Bolt non produce un punto, produce una distribuzione, e la mediana e'il punto che le si estrae.$$\mathrm{MAE} = \overline{|e|}, \qquad b = \bar e, \qquad V = \overline{(e - \bar e)^2},\qquad \mathrm{RMSE}^2 = b^2 + V .$$L'ultima e' un'identita', non un'approssimazione, e la cella la verifica numericamente. Va dettoche la decomposizione vale su MSE e **non** su MAE: "MAE = bias + varianza" non e' vera e non vienescritta da nessuna parte qui.**La weighted quantile loss** usa invece tutti e nove i quantili, ed e' la metrica con cui i lavorisu Chronos riportano i loro risultati:$$\mathrm{WQL} = \frac{\sum_i \sum_q 2\big[\,q\,(y_i - \hat y_{i,q})\mathbf 1\{y_i \ge \hat y_{i,q}\}+ (1-q)(\hat y_{i,q} - y_i)\mathbf 1\{y_i < \hat y_{i,q}\}\big]}{\sum_i |y_i|} .$$Numeratore e denominatore si sommano **su tutte le finestre prima di dividere**. Il rapporto perfinestra sarebbe instabile: di notte $\sum|y_i| \to 0$ e il quoziente esplode su finestre che noncontengono informazione.

In [ ]:
def metriche(pq, Y):
    """MAE, bias, varianza, RMSE e WQL da un blocco [N, Q, PRED] contro il vero [N, PRED]."""
    e = pq[:, QI_MED, :] - Y
    mae  = float(np.abs(e).mean()); bias = float(e.mean())
    var  = float(e.var(ddof=0));    mse  = float((e**2).mean())
    num = 0.0
    for k, q in enumerate(QUANTILI):
        d = Y - pq[:, k, :]
        num += 2.0*float(np.sum(np.where(d >= 0, q*d, (q - 1.0)*d)))
    den = float(np.abs(Y).sum())
    return dict(MAE=mae, bias=bias, Var=var, RMSE=float(np.sqrt(mse)),
                WQL=num/den if den > 0 else np.nan, _num=num, _den=den)

# verifica dell'identita' su ogni modello, non a campione
print("verifica RMSE^2 = bias^2 + Var")
_peggio = 0.0
for nome in PRED_Q:
    m = metriche(PRED_Q[nome], YTRUE)
    _peggio = max(_peggio, abs(m["RMSE"]**2 - (m["bias"]**2 + m["Var"]))/max(m["RMSE"]**2, 1e-12))
print(f"  scarto relativo massimo su {len(PRED_Q)} modelli: {_peggio:.3e}")
assert _peggio < 1e-9, "l'identita' non regge: c'e' un errore nel calcolo"

M = {nome: metriche(pq, YTRUE) for nome, pq in PRED_Q.items()}

## 6, Le tre tabelleUn confronto solo non separa le due cose che differiscono fra questi sedici checkpoint — il corpussu cui sono stati addestrati e la geometria con cui tokenizzano. Tre tabelle le separano.| | confronto | cosa varia | cosa isola ||---|---|---|---|| **A** | `chronos-bolt-tiny` pubblicato contro `p16-s16` retrainato | **solo** il corpus | l'effetto del preaddestramento || **B** | il pubblicato contro tutti e quindici | corpus **e** geometria | il divario complessivo verso lo stato dell'arte || **C** | `p16-s16` retrainato contro gli altri quattordici | **solo** la geometria | la domanda del progetto |**A** e' la tabella di controllo: stessa geometria $(16,16)$ da entrambe le parti, quindi tuttoquello che resta e' la differenza fra un corpus osservativo di ordine $10^7$ serie — che comprendeenergia e meteo — e la sola mixture sintetica del sweep. Il numero che esce da A e' la scalarispetto a cui vanno letti gli altri due.**B** e' il confronto che si legge per primo ma dice meno di quanto sembri: ogni riga sommal'effetto del corpus a quello della geometria, e senza A non c'e' modo di sapere quale dei due lasta muovendo.**C** e' l'unica delle tre che risponde alla domanda del progetto. Il riferimento e'`p16-s16-seed42`: stessa mixture, stessi centomila step, stesso seme, geometria stock di Bolt. Ilpubblicato e' escluso perche' non appartiene a quella popolazione. Uno scarto il cui intervallo noncontiene lo zero e' una differenza attribuibile alla tokenizzazione, e nient'altro.Le finestre sono disgiunte e distanti almeno $35$ ore, quindi la finestra **e'** l'unita'indipendente e un bootstrap appaiato su di essa e' legittimo senza blocchi ulteriori. Appaiatosignifica che a ogni ricampionamento tutti i modelli ricevono **le stesse** finestre: la differenzafra due modelli non viene sporcata dal fatto che ne hanno viste di diverse. Il verdetto e' a trevie — l'intervallo sta tutto sotto zero, tutto sopra, oppure lo contiene e i due modelli non sidistinguono su queste finestre.

In [ ]:
_rng_b = np.random.default_rng(SEED + 1)
BOOT_IDX = _rng_b.integers(0, N, size=(N_BOOT, N))     # gli stessi indici per tutti i modelli

def _mae_boot(pq):
    ae = np.abs(pq[:, QI_MED, :] - YTRUE).mean(1)       # MAE per finestra
    return ae[BOOT_IDX].mean(1)                          # [N_BOOT]

MAE_B = {nome: _mae_boot(pq) for nome, pq in PRED_Q.items()}
GEOM_DI = dict(RIGHE)

def _riga(nome, rif):
    g = GEOM_DI[nome]; m = M[nome]
    d = MAE_B[nome] - MAE_B[rif]
    lo, hi = float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))
    return dict(modello=nome, P=g[0] if g else np.nan, S=g[1] if g else np.nan,
                overlap=round(1 - g[1]/g[0], 3) if g else np.nan,
                MAE=m["MAE"], bias=m["bias"], Var=m["Var"], RMSE=m["RMSE"], WQL=m["WQL"],
                d_MAE=float(M[nome]["MAE"] - M[rif]["MAE"]), ic_lo=lo, ic_hi=hi,
                esito=("meglio" if hi < 0 else "peggio" if lo > 0 else "indistinguibile"))

def tabella(nomi, rif, titolo, file):
    righe = []
    for nome in nomi:
        r = _riga(nome, rif)
        if nome == rif:
            r.update(d_MAE=0.0, ic_lo=0.0, ic_hi=0.0, esito="(riferimento)")
        righe.append(r)
    T = pd.DataFrame(righe).sort_values("d_MAE").reset_index(drop=True)
    T.to_csv(f"{OUT_DIR}/{file}", index=False)
    print("\n" + "="*100); print(titolo); print("="*100)
    display(T.round(4))
    return T

_geom_tags = [t for t, g in RIGHE if g is not None]
_nota = "   [MODO PROVA - NUMERI NON REALI]" if MODO_PROVA else ""
print(f"canale {CANALE}   finestre {N}   bootstrap {N_BOOT}   intervalli al 95%{_nota}")
print("d_MAE < 0 = il modello sbaglia MENO del riferimento della sua tabella")

TAB_A = tabella([TAG_BASE, TAG_RIF], TAG_BASE,
                f"A. Il solo corpus: {BASE_ID} contro {TAG_RIF} retrainato  "
                "(stessa geometria da entrambe le parti)", "tab_A_corpus.csv")

TAB_B = tabella(_geom_tags, TAG_BASE,
                f"B. Corpus + geometria: i {len(_geom_tags)} retrainati contro il pubblicato  "
                "(le due cause sono sommate, non separate)", "tab_B_vs_pubblicato.csv")

TAB_C = tabella([t for t in _geom_tags if t != TAG_RIF] + [TAG_RIF], TAG_RIF,
                f"C. La sola geometria: gli altri {len(_geom_tags)-1} contro {TAG_RIF}  "
                "(stesso corpus, stessi step, stesso seme - il pubblicato e' escluso)",
                "tab_C_geometria.csv")

### Ripartizione per regimeIl $46\%$ circa dei target da 64 punti su questo canale e' interamente notturno, e una finestranotturna la prevede bene chiunque: mediata dentro il totale, quella meta' comprime le differenzefra modelli verso zero. La tabella principale resta su tutte le finestre — non si filtra insilenzio — ma la ripartizione qui sotto dice quanta parte del MAE viene da dove.

In [ ]:
_rows = []
for nome, pq in PRED_Q.items():
    ae = np.abs(pq[:, QI_MED, :] - YTRUE).mean(1)
    for reg in ["notte", "transizione", "giorno"]:
        sel = REGIME == reg
        if sel.sum() == 0: continue
        _rows.append(dict(modello=nome, regime=reg, finestre=int(sel.sum()),
                          MAE=float(ae[sel].mean()),
                          y_medio=float(YTRUE[sel].mean())))
REG = pd.DataFrame(_rows)
PIV = REG.pivot(index="modello", columns="regime", values="MAE").round(3)
REG.to_csv(f"{OUT_DIR}/per_regime.csv", index=False)
_cnt = {r: int((REGIME == r).sum()) for r in ["notte", "transizione", "giorno"]}
_ym  = {r: float(YTRUE[REGIME == r].mean()) for r in _cnt if (REGIME == r).any()}
print("MAE per regime (stesse finestre, stessa mediana)")
print("  finestre  " + "   ".join(f"{r}: {c}" for r, c in _cnt.items()))
print("  y medio   " + "   ".join(f"{r}: {v:.1f}" for r, v in _ym.items()))
display(PIV.sort_values("giorno"))

# MAE per orizzonte: costa nulla una volta che le previsioni sono in memoria, e dice se
# l'errore cresce liscio o a scalini lungo i 64 passi.
ORI = pd.DataFrame({nome: np.abs(pq[:, QI_MED, :] - YTRUE).mean(0)
                    for nome, pq in PRED_Q.items()})
ORI.index.name = "h"
ORI.to_csv(f"{OUT_DIR}/mae_per_orizzonte.csv")
print(f"\nMAE per orizzonte scritto in {OUT_DIR}/mae_per_orizzonte.csv  "
      f"(h=1: {ORI.iloc[0].min():.2f}-{ORI.iloc[0].max():.2f}, "
      f"h=64: {ORI.iloc[-1].min():.2f}-{ORI.iloc[-1].max():.2f})")

## 7, Come vanno lette**A da' la scala.** Finche' non si sa quanto vale il solo cambio di corpus su queste finestre, unoscarto di geometria non ha un metro. Se A vale molte unita' di MAE e gli scarti di C ne valgonofrazioni, la geometria e' un dettaglio rispetto ai dati di addestramento — che e' un risultato,non un fallimento, e va detto.**B non attribuisce.** Ogni riga somma le due cause. Se tutti e quindici stanno sopra ilpubblicato, la spiegazione piu' semplice e' il corpus, ed e' quella che A quantifica.**C e' la tabella del progetto**, ed e' l'unica su cui ha senso discutere di tokenizzazione.Tre cose che nessuna delle tre tabelle dice, e che conviene dichiarare prima che le chiedaqualcuno. E' un test su **un sito e un anno**: la popolazione e' una, e nulla qui la generalizza adaltre serie. E' un test **zero-shot** per i quindici, che la telemetria non l'hanno mai vista, manon e' detto che lo sia per il pubblicato, il cui corpus di preaddestramento comprende serieenergetiche e meteo e potrebbe contenere questo dataset o suoi parenti — un'altra ragione per nonleggere B come una classifica di merito. E le finestre sono centocinquanta: bastano per uno scartogrande, non per uno piccolo, e l'intervallo bootstrap e' li' per dire quale dei due si staguardando. Il verdetto *indistinguibile* non significa "uguali": significa che queste finestre nonbastano a separarli.

In [ ]:
print("="*78); print("RIEPILOGO"); print("="*78)

_a = TAB_A[TAB_A.modello == TAG_RIF].iloc[0]
print(f"A, solo corpus      {TAG_RIF} contro il pubblicato: "
      f"{_a.d_MAE:+.3f} MAE  [{_a.ic_lo:+.3f}, {_a.ic_hi:+.3f}]   {_a.esito}")

_mb = TAB_B[TAB_B.esito == "meglio"]; _pb = TAB_B[TAB_B.esito == "peggio"]
print(f"B, corpus+geometria  meglio del pubblicato {len(_mb)}/{len(TAB_B)}, "
      f"peggio {len(_pb)}/{len(TAB_B)}, indistinguibili "
      f"{len(TAB_B)-len(_mb)-len(_pb)}/{len(TAB_B)}")
if len(_mb): print("   meglio: " + ", ".join(_mb.modello))

_c = TAB_C[TAB_C.modello != TAG_RIF]
_mc = _c[_c.esito == "meglio"]; _pc = _c[_c.esito == "peggio"]
print(f"C, sola geometria    meglio di {TAG_RIF} {len(_mc)}/{len(_c)}, "
      f"peggio {len(_pc)}/{len(_c)}, indistinguibili {len(_c)-len(_mc)-len(_pc)}/{len(_c)}")
if len(_mc): print("   meglio: " + ", ".join(_mc.modello))
if len(_pc): print("   peggio: " + ", ".join(_pc.modello))

print(f"\nscala di riferimento: l'effetto del solo corpus vale {abs(_a.d_MAE):.3f} di MAE.")
print("Uno scarto di geometria molto piu' piccolo di questo e' un dettaglio; uno")
print("confrontabile o maggiore e' la cosa che il progetto sta cercando.")
print(f"\nscritti in {OUT_DIR}/: tab_A_corpus.csv, tab_B_vs_pubblicato.csv, tab_C_geometria.csv,")
print(f"                       per_regime.csv, mae_per_orizzonte.csv, finestre.csv, pred_*.npy")
if MODO_PROVA:
    print("\n*** MODO PROVA ATTIVO: i numeri vengono da un predittore finto, non dai modelli. ***")